# Stacks

## Stack Using Python Lists

In [1]:
# Stack Data Structure
class Stack():
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)				

    def pop(self):
        return self.items.pop()
    
    def is_empty(self):
        return self.items == []
    
    def peek(self):
        if not self.is_empty():
            return self.items[-1]
        
    def get_stack(self):
        return self.items

myStack = Stack()
myStack.push("A")
myStack.push("B")
myStack.push("C")
myStack.push("D")
print(myStack.peek())

D


## Stacks Using a Linked List

In [2]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class Stack:
    def __init__(self):
        self.top = None
        self.size = 0
    
    def push(self, value):
        new_node = Node(value)
        new_node.next = self.top
        self.top = new_node
        self.size += 1
    
    def pop(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        value = self.top.value
        self.top = self.top.next
        self.size -= 1
        return value
    
    def peek(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        return self.top.value
    
    def is_empty(self):
        return self.top is None

## Stack Using fixed type arrays, a lower abstraction

In [3]:
import ctypes

class ArrayStack:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.size = 0
        # Create a raw array using ctypes
        self._array = (ctypes.py_object * capacity)()
    
    def push(self, value):
        if self.size == self.capacity:
            self._resize(2 * self.capacity)
        self._array[self.size] = value
        self.size += 1
    
    def pop(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        self.size -= 1
        value = self._array[self.size]
        self._array[self.size] = None  # help garbage collection
        return value
    
    def peek(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        return self._array[self.size - 1]
    
    def _resize(self, new_capacity):
        new_array = (ctypes.py_object * new_capacity)()
        for i in range(self.size):
            new_array[i] = self._array[i]
        self._array = new_array
        self.capacity = new_capacity

## Speed Test

In [4]:
import ctypes
import time
import statistics

# Linked List Implementation
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedListStack:
    def __init__(self):
        self.top = None
        self.size = 0
    
    def push(self, value):
        new_node = Node(value)
        new_node.next = self.top
        self.top = new_node
        self.size += 1
    
    def pop(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        value = self.top.value
        self.top = self.top.next
        self.size -= 1
        return value
    
    def peek(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        return self.top.value


# Array-based Implementation using ctypes
class ArrayStack:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.size = 0
        self._array = (ctypes.py_object * capacity)()
    
    def push(self, value):
        if self.size == self.capacity:
            self._resize(2 * self.capacity)
        self._array[self.size] = value
        self.size += 1
    
    def pop(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        self.size -= 1
        value = self._array[self.size]
        self._array[self.size] = None
        return value
    
    def peek(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        return self._array[self.size - 1]
    
    def _resize(self, new_capacity):
        new_array = (ctypes.py_object * new_capacity)()
        for i in range(self.size):
            new_array[i] = self._array[i]
        self._array = new_array
        self.capacity = new_capacity


def benchmark_stack(stack_class, n_operations, n_runs=5):
    """Benchmark push, pop, and peek operations"""
    
    push_times = []
    pop_times = []
    peek_times = []
    mixed_times = []
    
    for _ in range(n_runs):
        # Benchmark Push
        stack = stack_class()
        start = time.perf_counter()
        for i in range(n_operations):
            stack.push(i)
        push_times.append(time.perf_counter() - start)
        
        # Benchmark Peek (on full stack)
        start = time.perf_counter()
        for _ in range(n_operations):
            stack.peek()
        peek_times.append(time.perf_counter() - start)
        
        # Benchmark Pop
        start = time.perf_counter()
        for _ in range(n_operations):
            stack.pop()
        pop_times.append(time.perf_counter() - start)
        
        # Benchmark Mixed Operations (push-push-pop pattern)
        stack = stack_class()
        start = time.perf_counter()
        for i in range(n_operations):
            stack.push(i)
            stack.push(i)
            stack.pop()
        mixed_times.append(time.perf_counter() - start)
    
    return {
        'push': {'mean': statistics.mean(push_times), 'std': statistics.stdev(push_times) if n_runs > 1 else 0},
        'pop': {'mean': statistics.mean(pop_times), 'std': statistics.stdev(pop_times) if n_runs > 1 else 0},
        'peek': {'mean': statistics.mean(peek_times), 'std': statistics.stdev(peek_times) if n_runs > 1 else 0},
        'mixed': {'mean': statistics.mean(mixed_times), 'std': statistics.stdev(mixed_times) if n_runs > 1 else 0},
    }


def main():
    test_sizes = [10_000, 100_000, 1_000_000]
    print("STACK IMPLEMENTATION BENCHMARK")
    for n in test_sizes:
        print(f"\n{'─' * 70}")
        print(f"Operations: {n:,}")
        print(f"{'─' * 70}")
        
        # Benchmark Linked List Stack
        ll_results = benchmark_stack(LinkedListStack, n)
        
        # Benchmark Array Stack
        arr_results = benchmark_stack(ArrayStack, n)
        
        # Print results
        print(f"\n{'Operation':<12} {'LinkedList (ms)':<20} {'Array (ms)':<20} {'Winner':<15}")
        print(f"{'-'*12} {'-'*20} {'-'*20} {'-'*15}")
        
        for op in ['push', 'pop', 'peek', 'mixed']:
            ll_time = ll_results[op]['mean'] * 1000
            arr_time = arr_results[op]['mean'] * 1000
            ll_std = ll_results[op]['std'] * 1000
            arr_std = arr_results[op]['std'] * 1000
            
            winner = "LinkedList" if ll_time < arr_time else "Array"
            speedup = max(ll_time, arr_time) / min(ll_time, arr_time)
            
            print(f"{op:<12} {ll_time:>8.2f} ± {ll_std:<8.2f} {arr_time:>8.2f} ± {arr_std:<8.2f} {winner} ({speedup:.2f}x)")
main()

STACK IMPLEMENTATION BENCHMARK

──────────────────────────────────────────────────────────────────────
Operations: 10,000
──────────────────────────────────────────────────────────────────────

Operation    LinkedList (ms)      Array (ms)           Winner         
------------ -------------------- -------------------- ---------------
push             2.01 ± 0.69         2.60 ± 0.12     LinkedList (1.30x)
pop              0.87 ± 0.23         0.84 ± 0.05     Array (1.04x)
peek             0.31 ± 0.08         0.49 ± 0.02     LinkedList (1.55x)
mixed            7.27 ± 7.83         5.05 ± 0.36     Array (1.44x)

──────────────────────────────────────────────────────────────────────
Operations: 100,000
──────────────────────────────────────────────────────────────────────

Operation    LinkedList (ms)      Array (ms)           Winner         
------------ -------------------- -------------------- ---------------
push            26.95 ± 2.31        39.75 ± 0.50     LinkedList (1.47x)
pop     

In [6]:
import ctypes
import time
import statistics

from cython_stack import CArrayStack, CLinkedListStack, CIntStack

# Pure Python Implementations
class Node:
    __slots__ = ('value', 'next')
    def __init__(self, value):
        self.value = value
        self.next = None

class PyLinkedListStack:
    def __init__(self):
        self.top = None
        self.size = 0
    
    def push(self, value):
        new_node = Node(value)
        new_node.next = self.top
        self.top = new_node
        self.size += 1
    
    def pop(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        value = self.top.value
        self.top = self.top.next
        self.size -= 1
        return value
    
    def peek(self):
        if self.top is None:
            raise IndexError("Stack is empty")
        return self.top.value


class PyArrayStack:
    def __init__(self, capacity=16):
        self.capacity = capacity
        self.size = 0
        self._array = (ctypes.py_object * capacity)()
    
    def push(self, value):
        if self.size == self.capacity:
            self._resize(self.capacity * 2)
        self._array[self.size] = value
        self.size += 1
    
    def pop(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        self.size -= 1
        value = self._array[self.size]
        self._array[self.size] = None
        return value
    
    def peek(self):
        if self.size == 0:
            raise IndexError("Stack is empty")
        return self._array[self.size - 1]
    
    def _resize(self, new_capacity):
        new_array = (ctypes.py_object * new_capacity)()
        for i in range(self.size):
            new_array[i] = self._array[i]
        self._array = new_array
        self.capacity = new_capacity

# Benchmark Functions
def benchmark_stack(stack_class, n_operations, n_runs=5, int_only=False):
    push_times = []
    pop_times = []
    peek_times = []
    
    for _ in range(n_runs):
        stack = stack_class()
        
        # Push
        start = time.perf_counter()
        for i in range(n_operations):
            stack.push(i)
        push_times.append(time.perf_counter() - start)
        
        # Peek
        start = time.perf_counter()
        for _ in range(n_operations):
            stack.peek()
        peek_times.append(time.perf_counter() - start)
        
        # Pop
        start = time.perf_counter()
        for _ in range(n_operations):
            stack.pop()
        pop_times.append(time.perf_counter() - start)
    
    return {
        'push': statistics.mean(push_times) * 1000,
        'pop': statistics.mean(pop_times) * 1000,
        'peek': statistics.mean(peek_times) * 1000,
        'total': (statistics.mean(push_times) + statistics.mean(pop_times) + statistics.mean(peek_times)) * 1000
    }


def main():
    print("STACK IMPLEMENTATION BENCHMARK: Pure Python vs Cython")
    
    implementations = [
        ("Python LinkedList", PyLinkedListStack, False),
        ("Python Array", PyArrayStack, False),
        ("Cython LinkedList", CLinkedListStack, False),
        ("Cython Array", CArrayStack, False),
        ("Cython Int-Only", CIntStack, True),
    ]
    
    test_sizes = [100_000, 1_000_000, 5_000_000]
    
    for n in test_sizes:
        print(f"\n{'─' * 90}")
        print(f"Operations: {n:,}")
        print(f"{'─' * 90}")
        
        results = {}
        for name, stack_class, int_only in implementations:
            results[name] = benchmark_stack(stack_class, n, n_runs=5, int_only=int_only)
        
        # Print header
        print(f"\n{'Implementation':<20} {'Push (ms)':<14} {'Pop (ms)':<14} {'Peek (ms)':<14} {'Total (ms)':<14} {'Speedup':<10}")
        print(f"{'-'*20} {'-'*14} {'-'*14} {'-'*14} {'-'*14} {'-'*10}")
        
        baseline = results["Python LinkedList"]['total']
        
        for name, _, _ in implementations:
            r = results[name]
            speedup = baseline / r['total']
            print(f"{name:<20} {r['push']:>10.2f}    {r['pop']:>10.2f}    {r['peek']:>10.2f}    {r['total']:>10.2f}    {speedup:>6.2f}x")
main()

STACK IMPLEMENTATION BENCHMARK: Pure Python vs Cython

──────────────────────────────────────────────────────────────────────────────────────────
Operations: 100,000
──────────────────────────────────────────────────────────────────────────────────────────

Implementation       Push (ms)      Pop (ms)       Peek (ms)      Total (ms)     Speedup   
-------------------- -------------- -------------- -------------- -------------- ----------
Python LinkedList         24.23          6.98          2.55         33.75      1.00x
Python Array              33.50          9.02          5.04         47.56      0.71x
Cython LinkedList          3.20          2.75          1.68          7.63      4.43x
Cython Array               1.82          2.25          1.67          5.74      5.88x
Cython Int-Only            2.04          2.60          2.67          7.31      4.62x

──────────────────────────────────────────────────────────────────────────────────────────
Operations: 1,000,000
───────────────────